In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
data_folder = Path("data_5000")

json_files = sorted(data_folder.glob("*.json"))

print("Total game files:", len(json_files))
assert len(json_files)==5000, "Expected exactly 5,000 new games in data_5000/"

Total game files: 5000


In [3]:
with open(json_files[0], "r") as f:
    game = json.load(f)

print(game.keys())

dict_keys(['schema_version', 'game', 'map', 'board', 'colors', 'player_state', 'buildings_by_color', 'resource_freqdeck', 'development_listdeck', 'random_state', 'action_records', 'num_turns', 'current_player_index', 'current_turn_index', 'current_prompt', 'is_initial_build_phase', 'is_discarding', 'discard_counts', 'is_moving_knight', 'is_road_building', 'free_roads_available', 'is_resolving_trade', 'current_trade', 'acceptees'])


In [4]:
for record in game["action_records"][:16]:
    print(record[0])

['ORANGE', 'BUILD_SETTLEMENT', 52]
['ORANGE', 'BUILD_ROAD', [23, 52]]
['RED', 'BUILD_SETTLEMENT', 29]
['RED', 'BUILD_ROAD', [29, 30]]
['BLUE', 'BUILD_SETTLEMENT', 17]
['BLUE', 'BUILD_ROAD', [17, 18]]
['WHITE', 'BUILD_SETTLEMENT', 31]
['WHITE', 'BUILD_ROAD', [30, 31]]
['WHITE', 'BUILD_SETTLEMENT', 46]
['WHITE', 'BUILD_ROAD', [45, 46]]
['BLUE', 'BUILD_SETTLEMENT', 12]
['BLUE', 'BUILD_ROAD', [11, 12]]
['RED', 'BUILD_SETTLEMENT', 20]
['RED', 'BUILD_ROAD', [0, 20]]
['ORANGE', 'BUILD_SETTLEMENT', 44]
['ORANGE', 'BUILD_ROAD', [40, 44]]


In [5]:
def extract_initial_placements(game):

    game_id = game["game"]["id"]
    colors = game["colors"]

    players = {}

    # Initialize player records
    for order, color in enumerate(colors, start=1):

        players[color] = {
            "Game_ID": game_id,
            "Player": color,
            "Placement_Order": order,
            "Settlement_1": None,
            "Road_1": None,
            "Settlement_2": None,
            "Road_2": None
        }

    # Only extract the first 16 actions
    actions = [
        record[0]
        for record in game["action_records"][:16]
    ]

    for color, action_type, value in actions:

        player = players[color]

        if action_type == "BUILD_SETTLEMENT":

            if player["Settlement_1"] is None:
                player["Settlement_1"] = value

            else:
                player["Settlement_2"] = value

        elif action_type == "BUILD_ROAD":

            if player["Road_1"] is None:
                player["Road_1"] = value

            else:
                player["Road_2"] = value

        else:
            raise ValueError(
                f"Unexpected setup action: {action_type}"
            )

    # Return four player records
    return list(players.values())

In [6]:
records = extract_initial_placements(game)

df_test = pd.DataFrame(records)

display(df_test)

,Game_ID,Player,Placement_Order,Settlement_1,Road_1,Settlement_2,Road_2
0,000a8611-8efa-43ef-aa17-2b695845f514,ORANGE,1,52,"[23, 52]",44,"[40, 44]"
1,000a8611-8efa-43ef-aa17-2b695845f514,RED,2,29,"[29, 30]",20,"[0, 20]"
2,000a8611-8efa-43ef-aa17-2b695845f514,BLUE,3,17,"[17, 18]",12,"[11, 12]"
3,000a8611-8efa-43ef-aa17-2b695845f514,WHITE,4,31,"[30, 31]",46,"[45, 46]"


In [7]:
all_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    records = extract_initial_placements(game)

    all_records.extend(records)

df = pd.DataFrame(all_records)

display(df.head(10))

,Game_ID,Player,Placement_Order,Settlement_1,Road_1,Settlement_2,Road_2
0,000a8611-8efa-43ef-aa17-2b695845f514,ORANGE,1,52,"[23, 52]",44,"[40, 44]"
1,000a8611-8efa-43ef-aa17-2b695845f514,RED,2,29,"[29, 30]",20,"[0, 20]"
2,000a8611-8efa-43ef-aa17-2b695845f514,BLUE,3,17,"[17, 18]",12,"[11, 12]"
3,000a8611-8efa-43ef-aa17-2b695845f514,WHITE,4,31,"[30, 31]",46,"[45, 46]"
4,00248db7-8590-49d2-a14c-aa3e61ff5128,RED,1,33,"[33, 34]",2,"[2, 3]"
5,00248db7-8590-49d2-a14c-aa3e61ff5128,ORANGE,2,21,"[21, 43]",47,"[45, 47]"
6,00248db7-8590-49d2-a14c-aa3e61ff5128,BLUE,3,53,"[52, 53]",49,"[49, 50]"
7,00248db7-8590-49d2-a14c-aa3e61ff5128,WHITE,4,11,"[11, 32]",28,"[28, 29]"
8,0035bf6c-f0d3-4308-9b1b-f2e2d8e2f24d,RED,1,30,"[29, 30]",4,"[3, 4]"
9,0035bf6c-f0d3-4308-9b1b-f2e2d8e2f24d,WHITE,2,9,"[8, 9]",1,"[1, 6]"


In [8]:
print("Total games:", df["Game_ID"].nunique())

print("Total records:", len(df))

print("Dataset shape:", df.shape)

Total games: 5000
Total records: 20000
Dataset shape: (20000, 7)


In [9]:
df.isna().sum()

Game_ID            0
Player             0
Placement_Order    0
Settlement_1       0
Road_1             0
Settlement_2       0
Road_2             0
dtype: int64

In [10]:
df.duplicated(
    subset=["Game_ID", "Player"]
).sum()

np.int64(0)

In [11]:
df.groupby("Game_ID").size().value_counts()

4    5000
Name: count, dtype: int64

In [12]:
for i in [1, 2]:

    valid = df.apply(
        lambda row: row[f"Settlement_{i}"]
        in row[f"Road_{i}"],
        axis=1
    )

    print(f"Round {i} invalid roads:", (~valid).sum())

Round 1 invalid roads: 0
Round 2 invalid roads: 0


In [13]:
map_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    game_id = game["game"]["id"]

    for tile in game["map"]["tiles"]:

        if tile["type"] == "LAND":

            map_records.append({
                "Game_ID": game_id,
                "Tile_ID": tile["id"],
                "Coordinate_X": tile["coordinate"][0],
                "Coordinate_Y": tile["coordinate"][1],
                "Coordinate_Z": tile["coordinate"][2],
                "Resource": tile["resource"],
                "Number": tile["number"]
            })

df_map = pd.DataFrame(map_records)

display(df_map.head(10))

,Game_ID,Tile_ID,Coordinate_X,Coordinate_Y,Coordinate_Z,Resource,Number
0,000a8611-8efa-43ef-aa17-2b695845f514,0,0,0,0,WOOD,11.0
1,000a8611-8efa-43ef-aa17-2b695845f514,1,1,-1,0,WOOD,10.0
2,000a8611-8efa-43ef-aa17-2b695845f514,2,0,-1,1,SHEEP,3.0
3,000a8611-8efa-43ef-aa17-2b695845f514,3,-1,0,1,WHEAT,6.0
4,000a8611-8efa-43ef-aa17-2b695845f514,4,-1,1,0,WOOD,5.0
5,000a8611-8efa-43ef-aa17-2b695845f514,5,0,1,-1,WHEAT,4.0
6,000a8611-8efa-43ef-aa17-2b695845f514,6,1,0,-1,SHEEP,9.0
7,000a8611-8efa-43ef-aa17-2b695845f514,7,2,-2,0,BRICK,5.0
8,000a8611-8efa-43ef-aa17-2b695845f514,8,1,-2,1,NaN,NaN
9,000a8611-8efa-43ef-aa17-2b695845f514,9,0,-2,2,ORE,8.0


In [14]:
output_folder = Path("processed_data_5000")

output_folder.mkdir(exist_ok=True)

df.to_csv(
    output_folder / "initial_placements.csv",
    index=False
)

df_map.to_csv(
    output_folder / "map_tiles.csv",
    index=False
)